# Scripting, Troubleshooting & LFCS Prep

## What's covered

- **Shell scripting essentials** — shebangs, variables (recap), conditionals (`if`, `test`, `[[ ]]`), loops (`for`/`while`/`until`), functions, exit codes
- **`set -euo pipefail`** — the production-script preamble; **`set -x`** for debugging
- **A complete example script** — putting the pieces together
- **Troubleshooting playbooks** — concrete diagnostic flows for the five problems you'll hit most:
  - Full disk
  - Out of memory
  - Runaway process
  - Network down
  - Service won't start
- **Mandatory Access Control** — **SELinux** (RHEL family) and **AppArmor** (Debian/Ubuntu) at the level LFCS tests
- **Performance monitoring** — `vmstat`, `iostat`, `sar` (from the `sysstat` package), and the patterns to look for
- **The LFCS exam itself** — format, what's tested, time strategy, hands-on tactics
- **What comes after LFCS** — where to go next (Docker, Kubernetes, Terraform, security-focused certs)

This is the capstone notebook. It assumes everything from notebooks 01-09 and pulls the threads together into the rhythms you'll use as a working Linux admin and on the exam itself.

## Why this final chapter exists

You've now covered every topic the LFCS exam tests. This notebook does three things the earlier chapters deferred:

1. **Wraps shell scripting** — you've used the shell interactively throughout; now we package what you've learned into reusable scripts. Every Linux admin job involves writing scripts.
2. **Gives troubleshooting muscle memory** — a sysadmin's value isn't memorising commands; it's having a *flow* for common problems. The playbooks here are the exact diagnostic flows you'll use.
3. **Prepares you for the exam itself** — LFCS is a hands-on exam (no multiple choice). Tactics matter as much as knowledge.

Treat this notebook as a reference. Come back to the playbooks when something breaks. Use the exam-prep section when you book your test date.

## Shell scripting essentials

A **shell script** is just a file with shell commands in it. The shell reads it line by line and runs each line as if you'd typed it.

### The shebang and making it executable

The first line of any executable script should be a **shebang**: `#!` followed by the path to the interpreter.

```bash
#!/bin/bash
# my first script
echo "Hello, $(whoami)!"
echo "Today is $(date +%A)."
```

Save as `hello.sh`. Two ways to run it:

```bash
bash hello.sh                # works without shebang or +x
./hello.sh                   # needs +x AND a shebang
chmod +x hello.sh            # one-time: make it executable
```

Why prefer `./hello.sh`? It treats your script like a real command. Put it in `~/bin` or `/usr/local/bin` and you can call it from anywhere.

**Common shebang variants:**

- **`#!/bin/bash`** — guaranteed bash (most distros). What you'll use 95% of the time.
- **`#!/bin/sh`** — POSIX-compliant shell (may be `dash`, `bash`, or `ash` depending on distro). More portable but no bash-only features.
- **`#!/usr/bin/env bash`** — finds bash in your `PATH`. Useful for scripts that should work on macOS too (where bash is in `/usr/local/bin/`).
- **`#!/usr/bin/env python3`** — Python script (same pattern).

### Variables (quick recap from notebook 02)

```bash
name="Alice"                  # no spaces around =
greeting="Hello, $name"       # double quotes expand $name
literal='Hello, $name'        # single quotes do NOT expand
echo "$greeting"              # always quote when reading
unset name                    # remove
```

### Script arguments

Inside a script, command-line arguments are available as numbered variables:

```bash
#!/bin/bash
echo "Script name: $0"
echo "First arg:   $1"
echo "Second arg:  $2"
echo "All args:    $@"        # individual quoted args
echo "All as one:  $*"        # all args as one string
echo "Number:      $#"        # count of args
echo "Exit of prev: $?"       # exit code of previous command
echo "PID of shell: $$"
echo "Last bg PID: $!"
```

Run `./script.sh apple banana cherry` and `$1` is `apple`, `$#` is `3`, etc.

**`"$@"` vs `$*`** — both expand to "all arguments", but `"$@"` (quoted) keeps each argument as its own word. **Always prefer `"$@"`** when forwarding arguments to another command, or you'll mangle arguments with spaces.

## Conditionals — `if`, `test`, `[[ ]]`

The basic `if` structure:

```bash
if COMMAND; then
    # runs if COMMAND exits 0 (success)
elif OTHER_COMMAND; then
    # runs if OTHER_COMMAND exits 0
else
    # runs if neither matched
fi
```

The condition is **any command**. Its **exit code** determines truth: `0` = true (success), non-zero = false. So `if ls /etc; then ...` runs the body when `ls /etc` succeeds.

### The `test` command (and its aliases `[` and `[[`)

The most common conditions use the `test` command, which has three equivalent forms:

```bash
test -f /etc/hostname             # the original
[ -f /etc/hostname ]              # alias (POSIX)
[[ -f /etc/hostname ]]            # bash extension (preferred in bash)
```

All three test "is `/etc/hostname` a regular file?" The `[[ ]]` form is the modern bash form — same behaviour, plus extras like `&&`, `||`, regex matching, and no need to quote variables.

```bash
if [[ -f "$config" ]]; then
    echo "config exists"
fi

if [[ "$count" -gt 10 ]]; then
    echo "too many"
fi
```

### The test operators worth memorising

**File tests:**

| Operator | True if |
|---|---|
| `-e FILE` | exists (any type) |
| `-f FILE` | regular file |
| `-d FILE` | directory |
| `-L FILE` | symbolic link |
| `-r FILE` | readable |
| `-w FILE` | writable |
| `-x FILE` | executable |
| `-s FILE` | exists and not empty |
| `FILE1 -nt FILE2` | newer than |
| `FILE1 -ot FILE2` | older than |

**String tests:**

| Operator | True if |
|---|---|
| `-z STR` | empty |
| `-n STR` | not empty |
| `STR1 = STR2` | equal (use `==` in `[[ ]]`) |
| `STR1 != STR2` | not equal |
| `STR1 < STR2` | comes before (alphabetical, inside `[[ ]]`) |
| `[[ STR =~ REGEX ]]` | regex match (bash extension) |

**Numeric tests:**

| Operator | True if |
|---|---|
| `-eq` | equal |
| `-ne` | not equal |
| `-lt` | less than |
| `-le` | less than or equal |
| `-gt` | greater than |
| `-ge` | greater than or equal |

**Combining:**

```bash
[[ -f "$file" && -r "$file" ]]                  # AND (use inside [[ ]])
[[ -f "$file" || -d "$file" ]]                  # OR
[[ ! -f "$file" ]]                              # NOT

# Short-circuit chains (work outside [[ ]] too)
[[ -d /backup ]] && echo "backup exists"        # only run if true
[[ -d /backup ]] || mkdir /backup               # only run if false
```

### `case` for multi-branch

When you have many alternatives, `case` is cleaner than chained `elif`:

```bash
case "$1" in
    start)
        echo "starting"
        ;;
    stop)
        echo "stopping"
        ;;
    status|info)                 # multiple patterns
        echo "alive"
        ;;
    *)                           # default
        echo "Usage: $0 {start|stop|status}"
        exit 1
        ;;
esac
```

Patterns use **glob** syntax (`*.log`, `[A-Z]*`), not regex. Use `;;` to end each branch.

## Loops — `for`, `while`, `until`

### `for` — over a list

```bash
for name in alice bob carol; do
    echo "Hello, $name"
done

for file in /etc/*.conf; do      # over a glob
    echo "Found: $file"
done

for i in {1..5}; do              # over a range
    echo "$i"
done

for i in $(seq 1 5); do          # equivalent
    echo "$i"
done
```

The C-style `for` works too — useful for arithmetic:

```bash
for ((i = 0; i < 10; i++)); do
    echo "$i"
done
```

### `while` — until a condition fails

```bash
count=0
while [[ $count -lt 5 ]]; do
    echo "count=$count"
    ((count++))
done

# Read a file line by line — the canonical pattern
while IFS= read -r line; do
    echo "Line: $line"
done < /etc/hostname
```

The `IFS= read -r line` form is the **correct** way to read a file line by line. `IFS=` keeps leading/trailing whitespace; `-r` prevents backslash mangling. Use this exact form.

### `until` — until a condition succeeds

```bash
# Wait for a service to start
until systemctl is-active nginx >/dev/null; do
    echo "waiting for nginx..."
    sleep 2
done
echo "nginx is up"
```

### Loop control — `break` and `continue`

```bash
for i in {1..10}; do
    [[ $i -eq 5 ]] && break       # exit the loop
    [[ $i -eq 3 ]] && continue    # skip to next iteration
    echo "$i"
done
```

## Functions, exit codes, and error handling

### Functions

```bash
greet() {
    local name="$1"               # local: doesn't leak to the rest of the script
    echo "Hello, $name"
}

greet "Alice"                     # call like a command
greet "Bob"
```

Two function definition styles:

```bash
name() { ... }                    # POSIX
function name { ... }             # bash-only
function name() { ... }           # also works in bash
```

Use the first form — most portable.

**Arguments** inside a function are the same as in scripts: `$1`, `$2`, `$@`, `$#`. They shadow the script's arguments inside the function.

**Use `local`** for any variable you don't want leaking out. Without it, every variable is global to the script — a source of subtle bugs.

### Exit codes

Every command in Linux returns an **exit code**: `0` = success, non-zero = failure. Different non-zero codes can mean different things — `man` pages document them.

```bash
ls /etc
echo "exit: $?"                   # always 0

ls /nonexistent
echo "exit: $?"                   # 2 (ENOENT)

# Set your script's exit code explicitly
exit 0                            # success
exit 1                            # generic failure
exit 2                            # by convention: misuse / bad arguments
```

In functions, use **`return`** instead of `exit` (which would kill the whole script):

```bash
is_root() {
    [[ $EUID -eq 0 ]]             # the `[[ ]]` itself returns 0 or 1
    return $?                     # propagate that (often omitted; the last command's exit is the function's)
}

if is_root; then
    echo "running as root"
fi
```

A function with no explicit `return` returns the exit code of its last command. So `is_root() { [[ $EUID -eq 0 ]]; }` works fine — the `[[ ]]` is the last command.

## `set -euo pipefail` — the production-script preamble

By default, bash is **forgiving** — a failing command doesn't stop the script, an unset variable expands to empty, a failing pipeline stage is swallowed. For production scripts, you want the opposite: **fail fast, fail loud**.

The canonical preamble for any non-trivial script:

```bash
#!/bin/bash
set -euo pipefail
IFS=$'\n\t'
```

Each flag explained:

- **`set -e`** ("errexit") — exit immediately if any command fails (returns non-zero). Without it, your script blunders on after a `cp` fails.
- **`set -u`** ("nounset") — error on referencing an unset variable. Catches typos like `$Path` (instead of `$PATH`) at runtime.
- **`set -o pipefail`** — a pipeline's exit code is the **last failing** command's, not just the last command's. Without it, `cmd-that-fails | grep something` returns success because `grep` succeeded.
- **`IFS=$'\n\t'`** — change the default field separator. Default is space + tab + newline; this drops space, which makes word-splitting safer when filenames contain spaces.

**One last trick**: if you genuinely want to allow a command to fail without exiting:

```bash
if ! some-command; then         # check explicitly
    echo "failed but okay"
fi

set +e                          # temporarily disable
risky-command
result=$?                       # save the exit code
set -e                          # re-enable
```

### Debugging with `set -x`

When a script misbehaves, **`set -x`** prints each command before running it (with variables expanded). Either set it at the top, or sprinkle it around problem areas:

```bash
set -x                         # debug on
do_something
set +x                         # debug off
```

You can also run a script in debug mode without editing it:

```bash
bash -x script.sh
```

`set -x` produces a lot of output. `set -xv` adds the original source line too. For long scripts, also consider `trap` for error reporting:

```bash
trap 'echo "FAIL at line $LINENO"' ERR
```

That fires on any failed command (when `-e` is on) and prints the line number.

## A complete example script

Putting it together — a script that takes a directory and reports the top-5 biggest files inside, with error handling:

```bash
#!/bin/bash
# top5.sh — report the 5 biggest files in a directory
set -euo pipefail

# --- usage ---
usage() {
    cat <<EOF
Usage: $0 <directory>
  Reports the 5 largest regular files under <directory>.
EOF
    exit 2
}

# --- argument check ---
if [[ $# -ne 1 ]]; then
    usage
fi

dir="$1"

if [[ ! -d "$dir" ]]; then
    echo "Error: '$dir' is not a directory" >&2
    exit 1
fi

# --- main ---
echo "Top 5 largest files under $dir:"
find "$dir" -type f -printf '%s %p\n' 2>/dev/null \
    | sort -rn \
    | head -5 \
    | while IFS= read -r size path; do
        printf "  %s  %s\n" "$(numfmt --to=iec --suffix=B "$size")" "$path"
    done

echo "done."
```

What it shows in one place:

- **Shebang + `set -euo pipefail`** — production-ready preamble.
- **`usage()`** function — common pattern.
- **Argument count check** with `$#`.
- **`[[ ! -d "$dir" ]]`** — file test with negation.
- **Error redirected to stderr** with `>&2`.
- **Pipeline** with `find` → `sort -rn` → `head -5` → `while read`.
- **`while IFS= read -r`** — the correct line-reading idiom.
- **`numfmt`** — useful tool for human-readable sizes.

Save, `chmod +x top5.sh`, run `./top5.sh /var/log`. Add `bash -x top5.sh /var/log` if anything misbehaves.

In [ ]:
# Try the script inline
!cat <<'EOF' > /tmp/top5.sh
#!/bin/bash
set -euo pipefail
[[ $# -ne 1 ]] && { echo "Usage: $0 <dir>" >&2; exit 2; }
dir="$1"
[[ -d "$dir" ]] || { echo "Not a dir: $dir" >&2; exit 1; }
find "$dir" -type f -printf '%s %p\n' 2>/dev/null | sort -rn | head -5
EOF
!chmod +x /tmp/top5.sh && /tmp/top5.sh /etc 2>/dev/null

## Troubleshooting methodology

A sysadmin's value isn't memorising commands — it's having a **flow**: a quick mental checklist that turns a vague "something's broken" into a specific diagnosis.

The general flow for every problem:

1. **Reproduce.** If you can't reproduce, you can't fix. Find the trigger.
2. **Narrow.** What changed? Recent deploy? New package? New user? Recent reboot? `journalctl --since "1 hour ago" -p err` is a great opener.
3. **Layer down.** Start at the highest abstraction the user sees, work down. Application logs → service status → system logs → kernel ring buffer.
4. **Bisect time.** When did it last work? `last`, `journalctl -b -1`, package manager history (`apt log`, `dnf history`).
5. **Isolate variables.** Test one factor at a time. If a service won't start, try running its binary by hand (with the same env). If you can't reach a host, ping its IP directly (bypassing DNS).
6. **Fix, then prevent.** Patch the symptom, then patch the cause. Then add monitoring so you see it before users do.

The next five sections are **playbooks** — concrete diagnostic flows for the five problems you'll meet most. Memorise the first commands of each.

## Playbook 1 — disk is full

**Symptom**: "No space left on device", services failing to write logs, packages failing to install.

**Step 1 — confirm and find which filesystem:**

```bash
df -h                                        # which filesystem is full?
df -i                                        # is it inodes instead of bytes?
```

If `Use%` is 100% on the relevant mount, you have a bytes problem. If `IUse%` is 100% on `df -i`, you have an inode problem (lots of tiny files, common on mail spools and build dirs).

**Step 2 — find the big stuff (bytes case):**

```bash
# Find biggest directories at one level under the full mount
sudo du -h --max-depth=1 /var 2>/dev/null | sort -h | tail -10

# Drill down
sudo du -h --max-depth=1 /var/log 2>/dev/null | sort -h | tail -10

# Find files over 100MB anywhere under /var
sudo find /var -type f -size +100M -printf '%s %p\n' 2>/dev/null | sort -rn | head
```

If you have it installed, **`ncdu /var`** is much faster — interactive TUI.

**Step 3 — check for deleted-but-held files** (the classic gotcha):

```bash
sudo lsof | grep deleted | head -20
```

When a process keeps a deleted file open, the disk space isn't freed until the process closes the file. Common culprits: services rotating their own logs without restart-and-reopen, an editor with an unlinked tempfile open.

Fix: restart the offending service, or `truncate -s 0` the file path inside `/proc/<pid>/fd/N` (advanced).

**Step 4 — check for inode exhaustion:**

If `df -i` shows 100%, you have too many files. Find directories with the most:

```bash
sudo find / -xdev -type d -exec sh -c 'echo "$(ls -A "$1" | wc -l) $1"' _ {} \; 2>/dev/null | sort -rn | head
```

Common culprits: PHP session dirs, mail queues, kernel `crash` dumps, abandoned package caches.

**Step 5 — quick wins:**

```bash
# Clear package manager caches
sudo apt clean                               # Debian/Ubuntu
sudo dnf clean all                           # RHEL/Fedora

# Trim systemd journals
sudo journalctl --vacuum-size=200M

# Remove old kernels (Ubuntu)
sudo apt autoremove --purge

# Remove old logs
sudo find /var/log -name "*.gz" -mtime +30 -delete

# Empty the trash (per user)
rm -rf ~/.local/share/Trash/*
```

## Playbook 2 — out of memory

**Symptom**: processes being killed unexpectedly, "Killed" in dmesg, sluggish system, swap thrashing.

**Step 1 — current state:**

```bash
free -h                                      # overall picture
cat /proc/meminfo | head -10                 # detailed
```

Look at `available` (not `free`) — that's what's actually allocatable. `free` underestimates because the page cache is counted as used (but reclaimable).

**Step 2 — what's using the memory?**

```bash
# Top memory consumers
ps -eo pid,user,rss,vsz,comm --sort=-rss | head -15

# With percentages
ps aux --sort=-%mem | head -15

# Interactive with sort by memory
top                                          # press M to sort by memory
```

`rss` (resident set size) is the honest "RAM in use by this process" number. **`vsz`** is virtual size — often much larger; ignore unless investigating address space leaks.

**Step 3 — check the OOM killer log:**

```bash
sudo dmesg -T | grep -i 'killed process\|out of memory'
sudo journalctl -k | grep -i 'killed process'
```

OOM killer output looks like `Out of memory: Killed process 12345 (java) total-vm:...`. That tells you what was killed and why.

**Step 4 — check swap usage:**

```bash
swapon --show                                # current swap
cat /proc/swaps

# If no swap configured, that's often the issue:
sudo fallocate -l 2G /swapfile
sudo chmod 600 /swapfile
sudo mkswap /swapfile
sudo swapon /swapfile
# Add to /etc/fstab for persistence
```

**Step 5 — protect critical processes:**

If a critical service keeps getting OOM-killed, give it a negative score adjustment:

```bash
# In its systemd unit:
[Service]
OOMScoreAdjust=-500                          # -1000 = never killed; 0 = default; +1000 = first to be killed
```

For LFCS-style emergency: **a swap file added at runtime** can buy you the breathing room to investigate. Long-term fix is usually "add more RAM" or "fix the leaking process."

## Playbook 3 — runaway process pinning the CPU

**Symptom**: load average climbing, fans spinning, system unresponsive.

**Step 1 — find the culprit:**

```bash
top                                          # press P to sort by CPU (default)
htop                                         # nicer interactive view

# One-shot snapshot
ps -eo pid,user,%cpu,%mem,comm --sort=-%cpu | head -10

# Watch CPU usage over time
top -b -n 1 | head -20                       # batch mode for scripting
```

**Step 2 — what's it actually doing?**

Once you have a PID, drill in:

```bash
sudo strace -p 12345                         # what syscalls is it making?
sudo strace -c -p 12345                      # summary after some seconds (Ctrl-C)
sudo lsof -p 12345                           # what files/sockets does it have open?
cat /proc/12345/status                       # detailed status
cat /proc/12345/cmdline | tr '\0' ' '       # the full command line
ls -l /proc/12345/cwd                        # where it's running from
```

`strace` is invaluable — a process spinning in a tight syscall loop (e.g. repeatedly opening the same file) tells you the bug instantly.

**Step 3 — control:**

```bash
# Lower its priority (doesn't kill it, just gives others more CPU)
sudo renice +10 -p 12345

# Limit its I/O priority
sudo ionice -c 3 -p 12345                    # "idle" class — only runs when nothing else needs the disk

# Pause it (great for "I want to inspect without it eating CPU")
sudo kill -STOP 12345
# ... investigate ...
sudo kill -CONT 12345                        # resume

# Kill it (escalate from polite to forceful)
kill 12345                                   # SIGTERM
sleep 5
kill -9 12345                                # SIGKILL if still alive
```

**Step 4 — if you can't kill it (state `D`):**

A process stuck in `D` state (uninterruptible sleep, usually waiting on disk or NFS) **cannot** be killed, not even with -9. Your options:

- Wait. If the underlying I/O eventually completes, the process unsticks.
- Fix the underlying cause (unmount the broken NFS share, etc.).
- Reboot.

## Playbook 4 — network is down

**Symptom**: can't reach external sites, slow DNS, SSH dropping.

This is the layered flow from notebook 08 — go through it methodically. Each step assumes the previous worked.

```bash
# Layer 1: do I have an interface and IP?
ip -br link                                  # any DOWN?
ip -br addr                                  # any address at all?

# Layer 2: can I reach my own gateway?
ip route                                     # what's my gateway?
ping -c 3 <gateway-IP>                       # respond?

# Layer 3: can I reach the wider internet by IP?
ping -c 3 1.1.1.1                            # bypasses DNS
ping -c 3 8.8.8.8

# Layer 4: does DNS resolve?
dig +short example.com
dig @8.8.8.8 +short example.com              # bypass local resolver

# Layer 5: where does the path die?
traceroute -n 1.1.1.1                        # which hop fails?
mtr 1.1.1.1                                  # continuous traceroute + ping

# Layer 6: is the target port reachable?
nc -zv example.com 443
curl -v https://example.com

# Layer 7 (on the server side): am I actually listening?
ss -tunlp                                    # all listening sockets, with process
sudo journalctl -u <service> -n 50           # recent service logs

# Last resort: see actual packets
sudo tcpdump -i any -n port 53               # in one shell
dig example.com                              # in another — should see the query go out
```

### Common immediate fixes

- **No IP** — restart NetworkManager: `sudo systemctl restart NetworkManager` or your interface: `sudo nmcli connection up "Wired connection 1"`.
- **DNS broken** — try a public resolver: edit `/etc/resolv.conf` to `nameserver 1.1.1.1` (or use `nmcli` to set DNS permanently).
- **Firewall blocking** — temporarily disable: `sudo ufw disable` (Debian) or `sudo systemctl stop firewalld` (RHEL). **Re-enable as soon as you're done diagnosing.**
- **DNS slow** — DNS server unreachable or returning slowly. Swap to `1.1.1.1` or `8.8.8.8`.

### Hardware troubleshooting

If `ip link` shows your interface as `DOWN` and `up` doesn't bring it up:

```bash
sudo dmesg | grep -i eth0                    # cable / link messages
sudo ethtool eth0                            # link speed, duplex, autoneg
sudo ethtool -S eth0                         # error counters (look for non-zero rx/tx errors)
```

## Playbook 5 — service won't start

**Symptom**: `systemctl start foo` returns an error, or `systemctl status` shows `failed`.

**Step 1 — read the status:**

```bash
sudo systemctl status foo
```

Most of the time, the last 10 log lines `status` shows already tell you the answer. Look for:
- `Permission denied` → file/dir/socket the service can't access.
- `Address already in use` → another process has the port.
- `No such file or directory` → wrong path, missing binary, missing config.
- `Failed to parse` → syntax error in a config file.

**Step 2 — full logs for this unit:**

```bash
sudo journalctl -u foo -n 100 --no-pager     # last 100 lines
sudo journalctl -u foo -f                    # follow live (in another window, then try start)
sudo journalctl -u foo --since "10 min ago"
```

If the service didn't even attempt to start, look at the systemd-level logs:

```bash
sudo journalctl -u foo -b                    # since boot
sudo systemctl list-units --failed           # any other failed services?
```

**Step 3 — test the config:**

Most services have a config-syntax check:

```bash
sudo nginx -t                                # nginx
sudo apachectl configtest                    # apache (also: httpd -t)
sudo sshd -t                                 # sshd
sudo named-checkconf                         # bind
```

If `-t` fails, you have a syntax error in the config. The error message includes the file and line.

**Step 4 — try running the binary by hand:**

If the service config looks fine but it still fails, run the binary directly with the same arguments systemd would use. Get the command from:

```bash
systemctl cat foo                            # the full unit including ExecStart
# Then run that command interactively:
sudo /usr/sbin/foo --foreground
```

You'll see all the errors directly. Often reveals environment/permission issues that systemd hid.

**Step 5 — check ports / files / SELinux:**

```bash
# Is the port already taken?
sudo ss -tunlp | grep :80

# Are the data directories the right owner/mode?
ls -ld /var/lib/foo /var/log/foo

# SELinux blocking? (RHEL/Fedora)
sudo ausearch -m AVC -ts recent              # recent SELinux denials
sudo getenforce                              # is it even enforcing?

# AppArmor blocking? (Debian/Ubuntu)
sudo aa-status
sudo dmesg | grep -i 'apparmor.*denied'
```

**Step 6 — daemon-reload?**

If you recently edited the unit file, did you `systemctl daemon-reload`? This is the most common "I changed it and it still doesn't work" cause.

### The general escalation path

For any service-won't-start:

```
status → journalctl -u → -t configtest → run binary directly → SELinux/AppArmor → daemon-reload
```

Master this sequence; it solves 90%+ of service issues.

## Mandatory Access Control — SELinux and AppArmor

The permissions you learned in notebook 06 (owner, group, other; `chmod`) are called **Discretionary Access Control (DAC)** — the file's owner decides who can access it. That's enough for most situations, but not enough for **defence in depth**: if a web server is compromised, normal DAC lets the attacker do everything the web server's user can do.

**Mandatory Access Control (MAC)** adds a second layer. The kernel checks a security policy *in addition to* the permissions. Even root is constrained. Two implementations matter on Linux:

| | SELinux | AppArmor |
|---|---|---|
| **Default on** | RHEL, CentOS, Fedora, Alma, Rocky | Ubuntu, openSUSE, Debian (optional) |
| **Model** | label every file/process; rules say which labels can interact | path-based profiles per program |
| **Complexity** | high, but powerful | simpler, easier to author |
| **Config in** | `/etc/selinux/`, contexts in xattrs | `/etc/apparmor.d/` |

For LFCS-level work, **know that they exist, how to check them, how to enable/disable, and how to read denial messages**. You don't need to author policies.

### SELinux at LFCS level

**Modes:**

```bash
getenforce                                   # current mode
sudo setenforce 0                            # set to Permissive (logs denials but doesn't block) — temporary
sudo setenforce 1                            # set to Enforcing — temporary
```

Three modes:
- **Enforcing** — denials are blocked. Production mode.
- **Permissive** — denials are *logged* but not blocked. Use for troubleshooting.
- **Disabled** — SELinux is off entirely (requires reboot to change; set in `/etc/selinux/config`).

**Persistent mode**: edit `/etc/selinux/config`:

```
SELINUX=enforcing                            # or permissive, or disabled
SELINUXTYPE=targeted                         # the policy ruleset; "targeted" is the standard
```

**Inspecting contexts** — every file and process has a "security context" (the label):

```bash
ls -lZ /etc/passwd                           # the -Z shows the SELinux context
ps -eZ | grep nginx                          # process contexts
```

A context looks like `system_u:object_r:passwd_file_t:s0` — `user:role:type:level`. The **type** field (`passwd_file_t`) is what the policy uses.

**Common SELinux fix**: a file in the wrong location has the wrong context. Restore it:

```bash
sudo restorecon -Rv /srv/web                 # restore default contexts under /srv/web
sudo chcon -t httpd_sys_content_t /srv/web/myfile    # set a specific type
```

**Reading denials**: SELinux denials show as **AVC** messages. Find recent ones:

```bash
sudo ausearch -m AVC -ts recent              # recent AVC denials
sudo journalctl _AUDIT_TYPE=AVC              # via journal
```

A tool called `audit2allow` can suggest policy modules to allow what was denied — useful for legitimate exceptions:

```bash
sudo ausearch -m AVC -ts recent | audit2allow -M mypolicy
sudo semodule -i mypolicy.pp
```

For LFCS, you mostly need: `getenforce`, `setenforce`, `/etc/selinux/config`, `ls -Z`, `restorecon`, `ausearch -m AVC`.

### AppArmor at LFCS level

AppArmor uses **per-program profiles** stored in `/etc/apparmor.d/`. Each profile says "this binary may access these paths with these permissions."

**Inspecting:**

```bash
sudo aa-status                               # all loaded profiles + their modes
```

Modes per profile:
- **enforce** — denials are blocked.
- **complain** — denials are logged but not blocked (like SELinux permissive).
- **disabled** — profile is loaded but inactive.

**Switch modes per profile:**

```bash
sudo aa-enforce /etc/apparmor.d/usr.sbin.nginx    # enforce mode for nginx
sudo aa-complain /etc/apparmor.d/usr.sbin.nginx   # complain mode (good for debugging)
sudo aa-disable /etc/apparmor.d/usr.sbin.nginx    # disable
```

**Find denials:**

```bash
sudo dmesg | grep -i apparmor
sudo journalctl -k | grep -i apparmor
```

The classic "I installed nginx but it can't read my files" issue on Ubuntu is often AppArmor restricting nginx to standard paths. Switch the profile to `complain`, repeat the action, see what was denied, then fix the profile.

For LFCS, the practical hands-on: know `aa-status`, `aa-enforce`/`aa-complain`/`aa-disable`, and where profiles live.

### A general MAC troubleshooting pattern

When a service should work but doesn't, and DAC permissions look correct:

1. **Suspect MAC.** Check `getenforce` (SELinux) or `aa-status` (AppArmor).
2. **Switch to permissive/complain mode** for the suspect.
3. **Retry the operation.** Does it work?
4. **If yes** — the policy was the problem. Look at the denials (`ausearch` or `dmesg`) to see what was being blocked, then write/adjust the policy.
5. **If no** — MAC isn't the issue. Continue investigating.

**Never permanently disable MAC** as the "fix". It defeats the security architecture. Use it to *confirm* MAC is the cause, then fix the policy.

## Performance monitoring — `vmstat`, `iostat`, `sar`

When a system is slow, you need numbers. Three tools, from the **`sysstat`** package (install with `sudo apt install sysstat` or `sudo dnf install sysstat`), give you the picture.

### `vmstat` — CPU, memory, swap, disk, system, in one line

```bash
vmstat 1                                     # every second, until you Ctrl-C
vmstat 1 5                                   # 5 samples, 1 second apart
vmstat -SM                                   # show memory in MB instead of KB
vmstat -d                                    # per-disk stats
```

The columns matter:

```
procs ---memory--- ---swap--- -----io---- -system-- ------cpu-----
 r  b   swpd  free  si   so   bi    bo   in   cs  us sy id wa st
```

Critical ones for diagnosis:

| Column | Meaning | Worry if... |
|---|---|---|
| **`r`** | runnable processes (in the run queue) | larger than CPU count for sustained time → CPU bottleneck |
| **`b`** | blocked processes (waiting for I/O) | persistently above 0 → I/O bottleneck |
| **`si`/`so`** | swap in/out (KB/s) | anything significant → swap thrashing (need more RAM) |
| **`bi`/`bo`** | blocks in/out (disk reads/writes per sec) | for size, look at iostat instead |
| **`us`/`sy`** | CPU spent on user / kernel | high `sy` often means thousands of syscalls — strace some processes |
| **`id`** | CPU idle | low + low `wa` = CPU-bound; low + high `wa` = I/O-bound |
| **`wa`** | CPU waiting on I/O | high → disk is the bottleneck |
| **`st`** | "stolen" (in a VM, time taken by the hypervisor) | high → noisy neighbour in cloud |

**The first row is averaged since boot** — usually not what you want. Look at subsequent rows for "right now."

### `iostat` — per-disk I/O detail

```bash
iostat -xz 1                                 # extended stats, hide idle devices, every second
iostat -xz 1 5
iostat -m                                    # MB/s instead of KB/s
```

`iostat -xz 1` is the go-to for "is my disk a bottleneck?" Key columns:

| Column | Meaning |
|---|---|
| **`r/s`, `w/s`** | read/write operations per second |
| **`rkB/s`, `wkB/s`** | KB read/written per second |
| **`await`** | average wait time per I/O (ms) — **the bottleneck signal**. Single-digit ms on SSDs, tens on spinning disks |
| **`r_await`, `w_await`** | read/write split |
| **`aqu-sz`** | average queue depth — sustained > 1 means I/O is queueing |
| **`%util`** | percentage of time the device was busy — 100% = saturated |

`await` is the killer metric. A normally-fast SSD with `await` jumping to hundreds of ms means I/O contention — possibly a failing disk, a backup running, or something hammering the FS.

### `sar` — long-term historical data

`sar` from the same `sysstat` package records performance snapshots every few minutes (configurable in `/etc/cron.d/sysstat` or `/etc/sysstat/sysstat`). Files end up in `/var/log/sysstat/` (one per day).

```bash
sar                                          # CPU stats for today
sar -r                                       # memory
sar -d                                       # disks
sar -n DEV                                   # network
sar -n DEV 1 5                               # network, live every second, 5 times
sar -f /var/log/sysstat/sa15                 # CPU on the 15th of this month (from saved data)
sar -s 09:00:00 -e 11:00:00                  # restrict to a time range
```

The value of `sar` is **post-incident analysis**: "we got paged at 3am — what was going on?" If sysstat was collecting, the answer is in `/var/log/sysstat/`.

To enable collection:

```bash
sudo systemctl enable --now sysstat          # or sysstat-collect.timer on newer distros
```

For LFCS: install `sysstat`, know `vmstat 1`, `iostat -xz 1`, and `sar -f`.

## The LFCS exam — what to expect

The Linux Foundation Certified System Administrator (LFCS) exam is a **2-hour, online, performance-based exam**. You're given a Linux machine via your browser; you SSH into it and solve real tasks. **No multiple choice. No essay questions. Just "do this on the live system."**

Important details (verify on the Linux Foundation site before booking, as they change):

- **Duration**: 2 hours
- **Distribution**: choose Ubuntu LTS or CentOS Stream when registering — the exam VM matches your choice
- **Passing score**: 66% (historically)
- **Format**: ~20-25 hands-on tasks. Each task has a defined "this works" criterion that's auto-checked.
- **Materials allowed**: you can use `man`, `info`, `--help`, and the live system's documentation. **No external sites, no offline notes.**
- **Retake**: one free retake within 12 months if you fail.
- **Validity**: 3 years from passing.

### Domains tested (with rough weights — verify current spec)

1. **Essential Commands** (~25%) — file ops, text processing, find, archives, basic shell
2. **Operation of Running Systems** (~20%) — boot, services, systemd, logs, time
3. **User and Group Management** (~10%) — useradd, usermod, sudo, password aging
4. **Networking** (~15%) — interfaces, routing, DNS, SSH, firewalls
5. **Service Configuration** (~10%) — basic web/DNS/email/database setup
6. **Storage Management** (~20%) — partitions, LVM, RAID, mounts, swap, quotas

Every domain has been covered across notebooks 01-09. Notebook 10 (this one) covers scripting, troubleshooting, and the cross-cutting bits.

### The exam VM environment

You get an SSH session into a stripped-down Ubuntu/CentOS box. **You can install whatever packages you need.** The exam wants to test that you can configure a real system, including pulling in tooling.

You'll have a terminal multiplexer (`tmux`) preinstalled — **use it**. Lose your SSH connection mid-exam and tmux saves you (your session continues on the server).

## Exam strategy and tactics

These are the tactics that turn knowledge into a pass.

### Before the exam

- **Practise on the same distro as the exam.** If you're taking Ubuntu, get to know `apt`, `ufw`, `netplan`. CentOS Stream candidates: `dnf`, `firewalld`, SELinux. Don't switch the week before.
- **Set up a practice VM.** Multipass or a cheap cloud droplet. Reset it often by spinning up fresh ones.
- **Do every "practise before moving on" exercise** in notebooks 01-09. The LFCS is hands-on; reading isn't enough.
- **Time yourself on individual tasks.** A LVM volume from scratch: 5 min. A working systemd service unit: 5 min. SSH-key login: 5 min. Most tasks should take well under 10 min.

### During the exam

- **Read every task fully before you start.** Tasks often have subtle constraints ("use UUID, not device name", "this user should not have a login shell"). Auto-graders check the exact outcome.
- **Use `man` shamelessly.** You're allowed. `man sshd_config`, `man crontab`, `man systemd.unit` are all available. Don't memorise; navigate.
- **`apropos` if you forget a command name.** `apropos compress`, `apropos partition`.
- **Save work after each task.** `:w` in vim, save in nano, etc. Don't leave files unsaved.
- **Skip and return.** If a task is taking too long, mark it (write the task number in a scratch file), move on, come back at the end.
- **Persistence matters.** If a task says "configure X to start at boot," you need to `systemctl enable`, not just `start`. If it says "make this permanent," `/etc/fstab`, `/etc/sysctl.conf`, `--permanent` for firewalld. Auto-graders check for persistence.
- **Use `tmux`** to avoid losing work on a dropped connection.
- **Don't reboot unless asked.** The grader may not wait for it. If you must, do it in the last few minutes.

### Mental checklist for any task

For each task, ask:

1. **Did I do it on the right host?** Some exams have multiple VMs.
2. **Did I do it as the right user?** Some tasks specify `sudo`; others want you to switch users.
3. **Is the change persistent across reboot?** Most tasks require yes.
4. **Did the auto-grader's test pass?** If you can verify yourself (`systemctl status`, `df`, `getent`, `ssh`), do.

### After the exam

- You get the result within 24 hours (sometimes immediately).
- If you pass: digital badge + PDF certificate.
- If you fail: review your weak domains, use the free retake.

## LFCS preparation checklist

A self-check before you book the exam. **You should be able to do each of these from a blank prompt, in under 5 minutes, without a notes**:

### Essentials
- [ ] Navigate the filesystem, use `find`, `grep`, `sed`, `awk` for ad-hoc queries
- [ ] Write a `for` loop and a `while read` loop
- [ ] Read and write a basic shell script with `set -euo pipefail`
- [ ] Use `tar -czf` / `tar -xzf` plus `zstd`/`xz` variants

### Permissions and users
- [ ] Create a user with home dir, set password, add to a group
- [ ] Configure `sudoers` for a specific user/group via `visudo -f`
- [ ] Set up SUID/SGID/sticky correctly
- [ ] Use `getfacl` / `setfacl`

### Storage
- [ ] Partition a new disk with `parted` (GPT)
- [ ] Create ext4 and xfs filesystems
- [ ] Mount permanently via `/etc/fstab` using UUID
- [ ] Configure a swap file
- [ ] Build LVM from scratch: PV → VG → LV → mkfs → mount → fstab
- [ ] Grow an LV + its filesystem (`lvextend -r`)
- [ ] Take and remove an LVM snapshot

### Services and boot
- [ ] Write a `.service` unit from scratch
- [ ] Write a `.timer` unit to schedule something
- [ ] Use `systemctl edit` for a drop-in
- [ ] Query `journalctl` by unit, time, priority, boot
- [ ] Configure a cron job

### Network
- [ ] Configure a static IP via `nmcli`
- [ ] Configure DNS
- [ ] Set up SSH key-based login
- [ ] Configure SSH server (key-only, restrict users)
- [ ] Configure a firewall to allow specific ports (`ufw` or `firewalld`)
- [ ] Diagnose with `ss`, `dig`, `traceroute`, `tcpdump`

### Packages and updates
- [ ] Install/remove/search via `apt` or `dnf`
- [ ] Add a third-party repo
- [ ] Use `dnf history` or `apt log` to see recent changes

### Logs and monitoring
- [ ] Find errors with `journalctl -p err --since`
- [ ] Find a service's logs with `journalctl -u`
- [ ] Find what's eating disk: `du -h --max-depth=1`
- [ ] Find what's eating CPU/memory: `ps`, `top`
- [ ] Read `vmstat`/`iostat` output

### MAC
- [ ] Switch SELinux between enforcing/permissive (`setenforce`)
- [ ] Restore SELinux contexts (`restorecon -Rv`)
- [ ] Check AppArmor status (`aa-status`)

If most of these are "yes, in 5 minutes", you're ready. If many are "I'd need to look it up", do another round of hands-on practice on the weak areas.

## After LFCS — where to go next

LFCS gets you to "I can administer a Linux server competently." From here, the field branches in several directions — pick what aligns with your goals.

### The natural next steps

| Direction | What you'd learn | Common certs |
|---|---|---|
| **Containers** | Docker, container fundamentals, image building, multi-container setups | (no widely-recognised Docker-only cert; the `docker` repo in this project takes you here) |
| **Container orchestration** | Kubernetes, pods, services, deployments, networking, storage | **CKA** (Certified Kubernetes Administrator), CKAD (Developer), CKS (Security) |
| **Infrastructure as Code** | Declarative cloud provisioning | **HashiCorp Terraform Associate**, AWS/Azure/GCP cloud certs |
| **Cloud platforms** | AWS / Azure / GCP services, identity, networking | **AWS Solutions Architect**, **Azure Administrator**, **GCP Associate Engineer** |
| **Linux deeper** | Advanced kernel internals, performance tuning, custom builds | **LFCE** (Linux Foundation Certified Engineer) — LFCS's senior sibling |
| **Security** | Pentesting, hardening, SOC operations | OSCP, CompTIA Security+, CISSP (later) |
| **DevOps** | CI/CD pipelines, observability, GitOps | various — Jenkins, GitHub Actions, ArgoCD |
| **SRE** | Reliability engineering, SLOs, incident response | Google SRE books; no canonical cert |

### A sensible "next 12 months" path for a sysadmin → DevOps direction

1. **Docker** (1 month) — containers, images, Dockerfile, compose. Strongly complements LFCS.
2. **Kubernetes basics + CKA** (2-3 months) — the dominant orchestration platform; CKA is hands-on like LFCS.
3. **Terraform + AWS Solutions Architect Associate** (3 months) — IaC + a cloud platform.
4. **Pick a depth**: Kubernetes Security (CKS), Cloud-specialist deeper certs, or SRE-style observability/reliability.

### A sensible "next 12 months" path for staying on the systems side

1. **LFCE** (3 months) — deeper Linux: clustering, advanced storage, advanced networking, IPv6.
2. **Configuration management** — Ansible (very LFCS-adjacent; `ansible-playbook` is the natural extension of writing shell scripts). RHCSA/RHCE if you want vendor-specific.
3. **Performance tuning** — Brendan Gregg's *Systems Performance* is the canonical text.
4. **Specialise**: a database (PostgreSQL admin), a service mesh, observability (Prometheus, Grafana, OpenTelemetry).

### A few books worth reading whatever direction you go

- **"How Linux Works" — Brian Ward** — the friendly companion to this curriculum, deeper on internals.
- **"The Linux Command Line" — William Shotts** — free, comprehensive, classic.
- **"Linux System Programming" — Robert Love** — what's under the hood of every command.
- **"Site Reliability Engineering" — Google (free online)** — how production systems are actually run at scale.

## Closing the loop — the curriculum recap

You've worked through ten notebooks. A quick map of what's now in your head:

| # | Topic | Key takeaway |
|---|---|---|
| 01 | Getting Started | Linux model, getting a prompt, basic commands, FHS, help |
| 02 | Shell Mechanics | Variables, env, quoting, redirection, pipes, globs, expansion order |
| 03 | Files & Permissions | File ops, links, the seven file types, `chmod`/`chown`/`umask` |
| 04 | Text Processing & Find | The Unix toolkit + `find` + the top-N idiom |
| 05 | Processes, Jobs & Signals | `ps`/`top`/`htop`, jobs, signals, scheduling, cron + timers preview |
| 06 | Users, Groups & Access Control | `/etc/passwd`/`shadow`/`group`, sudo, SUID/SGID/sticky, ACLs, PAM |
| 07 | Storage, Filesystems & Mounts | Block devices, partitions, ext4/xfs, fstab, LVM, RAID, quotas |
| 08 | Networking & SSH | `ip`, DNS, SSH (keys, config, agent), rsync, firewall, troubleshooting |
| 09 | Services, Boot, systemd & Packages | Boot stages, systemd units, `systemctl`, `journalctl`, apt/dnf |
| 10 | Scripting, Troubleshooting & LFCS | Scripts, playbooks, MAC, sysstat, the exam itself |

**You now have the toolkit to run a Linux server.** That's a real skill — one a huge fraction of the working world depends on.

The next step is hands-on time. Spin up a VM and *break things on purpose*. Fill the disk and recover. OOM a process and tune the policy. Misconfigure SSH and lock yourself out from one window, then fix it from another. Set up a service from scratch and write its systemd unit and timer. You'll learn ten times more from one well-investigated breakage than from re-reading a chapter.

Good luck on the exam — and welcome to Linux.